# Multimodal RAG with litesearch

This notebook demonstrates the full RAG pipeline:

1. **Hierarchical PDF ingest** — parent/child chunks via `hierarchical_pdf_texts()`
2. **Image extraction with captions** — PDF-embedded captions via regex (`pdf_images_with_captions()`)
3. **Hybrid search** — FTS5 + vector RRF over text and image records
4. **Agentic RAG** — `LiteRAG.query()` with `search_knowledge_base` tool
5. **Reranking** — optional `FastRerank` for precision boost

**Dataset**: Three open arXiv ML papers with diagrams and tables:
- *Attention Is All You Need* (1706.03762) — transformer architecture diagrams
- *BERT* (1810.04805) — pre-training figures
- *An Image is Worth 16x16 Words / ViT* (2010.11929) — vision figures

> Requires `pip install litesearch lisette chonkie` and either Ollama (`qwen2.5`) or a cloud API key.
> Reranking requires `pip install flashrank`.

## 1. Setup

In [ ]:
import os, urllib.request
from pathlib import Path
from fastcore.all import *
from litesearch import *
from litesearch.data import hierarchical_pdf_texts, pdf_images_with_captions
from litesearch.rag import LiteRAG
from litesearch.utils import FastRerank

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Model for embedding — lightweight, CPU-only static model
ENCODER_MODEL = 'minishlab/potion-retrieval-32M'

# LLM for generation — change to any LiteLLM string
# Examples: 'ollama/qwen2.5', 'claude-haiku-4-5-20251001', 'gpt-4o-mini'
LLM_MODEL = 'ollama/qwen2.5'

## 2. Download PDFs

In [ ]:
pdf_dir = Path('pdfs_rag_demo')
pdf_dir.mkdir(exist_ok=True)

papers = {
    'attention.pdf': 'https://arxiv.org/pdf/1706.03762',
    'bert.pdf':      'https://arxiv.org/pdf/1810.04805',
    'vit.pdf':       'https://arxiv.org/pdf/2010.11929',
}

for name, url in papers.items():
    dest = pdf_dir / name
    if not dest.exists():
        print(f'Downloading {name}...')
        urllib.request.urlretrieve(url, dest)
    else:
        print(f'{name} already present')

pdf_paths = sorted(pdf_dir.glob('*.pdf'))
print(f'\nReady: {[p.name for p in pdf_paths]}')

## 3. Ingest — Hierarchical Text Chunks

Each page's markdown is split into **parent chunks** (512 tokens) for LLM context and **child chunks** (128 tokens) for precise retrieval. The hybrid FTS5+vector search operates on child chunks; `expand_parents=True` in `LiteRAG` fetches the parent for richer context.

In [ ]:
DB_PATH = 'rag_demo.db'
db = database(DB_PATH)
store = db.get_store()
print('Store columns:', [c.name for c in store.columns])

In [ ]:
from chonkie import AutoEmbeddings
_enc = AutoEmbeddings().get_embeddings(ENCODER_MODEL)
encoder = lambda texts: _enc.embed(texts)

In [ ]:
# Only ingest if store is empty
if len(store()) == 0:
    for pdf_path in pdf_paths:
        print(f'Ingesting text: {pdf_path.name}...')
        chunks = hierarchical_pdf_texts(str(pdf_path), use_parent_child=True,
                                        parent_size=512, child_size=128)
        store.ingest_chunks(chunks, encoder=encoder)
        print(f'  → {len([c for c in chunks if c.get("hierarchy_level")==1])} parents, '
              f'{len([c for c in chunks if c.get("hierarchy_level")==2])} children')

text_count = len(store())
print(f'\nTotal text records: {text_count}')

## 4. Ingest — Images with PDF-Embedded Captions

Each image is paired with its figure caption extracted directly from the PDF text (e.g. *"Figure 1: The Transformer model architecture."*). No VLM or internet needed — the caption is already in the PDF.

Image records have `chunk_type='image'`; the caption is stored as `content` and the image bytes are base64-encoded in `metadata`.

In [ ]:
# Ingest image records (idempotent — skip if already present)
existing_img = len([r for r in store() if r.get('chunk_type') == 'image'])

if existing_img == 0:
    for pdf_path in pdf_paths:
        print(f'Extracting images: {pdf_path.name}...')
        img_records = pdf_images_with_captions(str(pdf_path))
        # Embed captions so image records are searchable via semantic query
        captions = [r['content'] for r in img_records]
        if captions:
            embs = encoder(captions)
            for r, e in zip(img_records, embs):
                r['embedding'] = e.tobytes()
        store.insert_all(img_records)
        print(f'  → {len(img_records)} images')
        for r in img_records:
            print(f'    caption: {r["content"][:80]}')

total_img = len([r for r in store() if r.get('chunk_type') == 'image'])
print(f'\nTotal image records: {total_img}')

## 5. Direct Search Queries

Hybrid search works across both text and image records (since image captions are embedded as text).

In [ ]:
from litesearch.data import pre

def search_demo(query, limit=5):
    emb = encoder([query])[0].tobytes()
    results = db.search(pre(query), emb, limit=limit) or []
    print(f'Query: "{query}" → {len(results)} results')
    for r in results:
        ctype = r.get('chunk_type', 'text')
        lvl   = r.get('hierarchy_level', 0)
        score = r.get('_rrf_score', 0)
        print(f'  [{ctype}/lvl{lvl}] score={score:.4f} | {r["content"][:100]}')

search_demo('multi-head attention mechanism')

In [ ]:
# Cross-modal: text query hitting image captions
search_demo('architecture diagram transformer encoder decoder')

In [ ]:
# Compare flat vs hierarchical — fetch a child, then expand to its parent
emb = encoder(['positional encoding sinusoidal'])[0].tobytes()
child_hits = db.search(pre('positional encoding'), emb, limit=3) or []
print('Child hits:')
for h in child_hits:
    print(f'  id={h.get("id")} lvl={h.get("hierarchy_level")} parent_id={h.get("parent_id")} | {h["content"][:80]}')

# Expand to parent
parent_ids = [h['parent_id'] for h in child_hits if h.get('parent_id')]
if parent_ids:
    ph = ','.join('?' * len(parent_ids))
    parents = list(db.q(f'SELECT id, content, hierarchy_level FROM store WHERE id IN ({ph})', parent_ids))
    print('\nExpanded parents:')
    for p in parents:
        print(f'  id={p["id"]} lvl={p["hierarchy_level"]} | {p["content"][:120]}')

## 6. Reranking with FastRerank

Optional cross-encoder reranking improves precision. Uses CPU-only ONNX — no GPU, no PyTorch.

In [ ]:
rr = FastRerank(model='ms-marco-MiniLM-L-12-v2', max_length=160)

query = 'How does scaled dot-product attention work?'
emb = encoder([query])[0].tobytes()
candidates = db.search(pre(query), emb, limit=10) or []

# Rerank: top 3
docs = [{'content': r.get('content',''), **r} for r in candidates]
reranked = rr.rerank(query, docs, top_n=3)

print(f'Query: "{query}"')
print('\nTop 3 after reranking:')
for i, r in enumerate(reranked, 1):
    print(f'  [{i}] {r["content"][:120]}')

## 7. Agentic RAG with LiteRAG

`LiteRAG` wraps everything: retrieval → context → Lisette Chat with `search_knowledge_base` tool.

In [ ]:
rag = LiteRAG(
    db_path=DB_PATH,
    encoder_model=ENCODER_MODEL,
    llm_model=LLM_MODEL,
    expand_parents=True,
    top_n=4,
    search_level=None,  # set to 'm' to also allow web search fallback
)
print('LiteRAG ready:', rag.llm_model)

In [ ]:
# Basic query — LLM will call search_knowledge_base first
response = rag.query('Explain the multi-head attention mechanism and why it is useful.', max_steps=4)
print(response.choices[0].message.content)

In [ ]:
# Cross-paper comparison
response = rag.query('What are the key architectural differences between BERT and the original Transformer?', max_steps=5)
print(response.choices[0].message.content)

In [ ]:
# Image-grounded query — hits image caption records
response = rag.query('What does Figure 1 in the Attention paper show?', max_steps=3)
print(response.choices[0].message.content)

### LiteRAG with Reranker

In [ ]:
rag_rr = LiteRAG(
    db_path=DB_PATH,
    encoder_model=ENCODER_MODEL,
    llm_model=LLM_MODEL,
    expand_parents=True,
    k=10,
    top_n=3,
    reranker=FastRerank(model='ms-marco-MiniLM-L-12-v2', max_length=160),
    search_level=None,
)
response = rag_rr.query('How does ViT handle image patches differently from CNNs?', max_steps=4)
print(response.choices[0].message.content)

### Streaming

In [ ]:
print('Streaming response:')
for chunk in rag.query('Summarise the positional encoding approach in the Transformer.', stream=True, max_steps=3):
    delta = getattr(chunk.choices[0].delta, 'content', '') or ''
    print(delta, end='', flush=True)
print()

### Async Query

In [ ]:
import asyncio

async def run_async():
    result = await rag.aquery('Summarise section 3 of the Attention paper.', max_steps=3)
    print(result.choices[0].message.content)

await run_async()

### Extra Tools

In [ ]:
# Plain Python function — no @tool decorator needed
def count_papers(topic: str) -> str:
    "Count how many papers in the DB mention a topic."
    results = db.search(topic, encoder([topic])[0].tobytes(), limit=50) or []
    sources = set()
    for r in results:
        try: sources.add(json.loads(r.get('metadata') or '{}').get('source',''))
        except: pass
    return f'{len(sources)} paper(s) mention "{topic}": {list(sources)}'

rag_multi = LiteRAG(
    db_path=DB_PATH,
    encoder_model=ENCODER_MODEL,
    llm_model=LLM_MODEL,
    tools=[count_papers],
    search_level=None,
)
response = rag_multi.query('How many papers in this collection discuss attention mechanisms?', max_steps=4)
print(response.choices[0].message.content)

## 8. GraphRAG (Phase 4 — optional)

Extracts (subject, predicate, object) triples from context and stores them in `graph_triples`.

In [ ]:
from litesearch.rag import make_explore_relationships_tool
import json

graph_tool = make_explore_relationships_tool(db, LLM_MODEL)

rag_graph = LiteRAG(
    db_path=DB_PATH,
    encoder_model=ENCODER_MODEL,
    llm_model=LLM_MODEL,
    tools=[graph_tool],
    search_level=None,
)
response = rag_graph.query(
    'Extract the key relationships between Transformer components from the Attention paper.',
    max_steps=5)
print(response.choices[0].message.content)

# Inspect stored triples
if 'graph_triples' in db.table_names():
    triples = list(db.q('SELECT subject, predicate, object FROM graph_triples LIMIT 10'))
    print(f'\nStored {len(triples)} triples:')
    for t in triples: print(f'  ({t["subject"]}) --[{t["predicate"]}]--> ({t["object"]})')

## Cleanup

In [ ]:
# Optionally remove the demo DB
# import os
# os.remove(DB_PATH)